# wandb-watch-model — ex2: auto-watch every submodule with at least one trainable own-parameter

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `wandb-watch-model`. Running the final beacon cell reports progress against the `Logging: wandb.watch model` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.watch model` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-watch-model`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-watch-model"
DD_SUBTOPIC = "Logging: wandb.watch model"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `wandb.watch` only the trainable submodules

Ex1 watched a fixed `model.out_layers[-1]`. The deepening move: AUTO-DETECT which submodules to watch by checking `any(p.requires_grad for p in m.parameters(recurse=False))`. This way a fine-tune script that freezes/unfreezes layers at runtime still watches the right thing without code edits.

```python
watched = []
for qname, m in model.named_modules():
    if qname == '':
        continue
    own = list(m.parameters(recurse=False))
    if own and any(p.requires_grad for p in own):
        wandb.watch(m, log='all', log_freq=K)
        watched.append(qname)
```

**`recurse=False` is load-bearing.** Without it, every ancestor of a trainable leaf shows up as 'has a trainable param' and you double-hook. `recurse=False` gives only the params OWNED by the module itself.

**Skip the root.** `named_modules()` yields `('', model)` first — same filter as the named-modules report pattern.

### Exercise 2 — auto-watch every submodule with at least one trainable own-parameter

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a model by walking `named_modules()` and watch (via `wandb.watch`) only the submodules that own at least one trainable parameter, returning the sorted list of watched qnames.
> Keywords: wandb, watch, named-modules, trainable, requires_grad
> ```

**KCs targeted:** `named-modules-with-recurse-false-params`, `trainable-param-filter-any`

Implement `ex2_watch_trainable_modules(model, log_freq)`. The auto-watch generalization of ex1.

Algorithm:
1. Walk `model.named_modules()`.
2. SKIP entries where `qname == ''` (the root container).
3. For each remaining `(qname, m)`:
   - Get `own = list(m.parameters(recurse=False))` — params OWNED by `m`, not its children.
   - If `own` is empty: skip (no parameters of its own — it's a wrapper like `nn.Sequential`).
   - If `any(p.requires_grad for p in own)`: call `wandb.watch(m, log='all', log_freq=log_freq)` AND append `qname` to a `watched` list.
   - Else (all own params are frozen): skip (don't watch frozen layers — they produce zero-gradient histograms).
4. Return `sorted(watched)`.

Inputs:
- `model`: `nn.Module`.
- `log_freq`: int.

Output: sorted `list[str]` of watched qnames.

The test mocks `wandb.watch` to verify exactly which modules got hooked + that `log='all'` + `log_freq` are forwarded correctly.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb
import torch.nn as nn

def ex2_watch_trainable_modules(model: nn.Module, log_freq: int) -> list:
    """Watch every submodule with at least one trainable own-param; return sorted qnames."""
    raise NotImplementedError()


def _test_ex2():
    import torch.nn as nn
    import sys
    from unittest.mock import MagicMock
    if 'wandb' in sys.modules and not isinstance(sys.modules['wandb'], MagicMock):
        del sys.modules['wandb']
    sys.modules.setdefault('wandb', MagicMock())
    import wandb

    # === Build a small model: encoder frozen, head trainable ===
    class Head(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(16, 32)
            self.fc2 = nn.Linear(32, 10)

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Linear(8, 16),
                nn.ReLU(),
                nn.Linear(16, 16),
            )
            self.head = Head()
            # Freeze the encoder.
            for p in self.encoder.parameters():
                p.requires_grad_(False)

    wandb.watch.reset_mock()
    model = Net()
    watched = ex2_watch_trainable_modules(model, log_freq=50)

    # === Watched list is sorted and contains exactly the trainable-owning leaves ===
    assert isinstance(watched, list), f'must return list, got {type(watched).__name__}'
    assert watched == sorted(watched), f'must be sorted, got {watched}'
    # Encoder's Linear children OWN params but they're frozen → not watched.
    # ReLU has no params → not watched.
    # Sequential is a wrapper with no own params → not watched.
    # Head itself has no own params (its fcN do) → not watched.
    # So watched = ['head.fc1', 'head.fc2'].
    assert watched == ['head.fc1', 'head.fc2'], f'watched qnames wrong: {watched}'

    # === wandb.watch called exactly len(watched) times ===
    assert wandb.watch.call_count == 2, f'expected 2 wandb.watch calls, got {wandb.watch.call_count}'

    # === Each call uses log='all' and the passed log_freq ===
    for call in wandb.watch.call_args_list:
        assert call.kwargs.get('log') == 'all', f"log must be 'all', got {call.kwargs.get('log')!r}"
        assert call.kwargs.get('log_freq') == 50, f'log_freq must propagate, got {call.kwargs.get("log_freq")!r}'
        # Positional arg = the module itself.
        assert isinstance(call.args[0], nn.Linear), f'must watch the actual leaf module, got {type(call.args[0]).__name__}'

    # === Unfreezing the encoder grows the watched set ===
    wandb.watch.reset_mock()
    for p in model.encoder.parameters():
        p.requires_grad_(True)
    watched2 = ex2_watch_trainable_modules(model, log_freq=100)
    # Encoder Linears (encoder.0, encoder.2) now also trainable.
    assert watched2 == ['encoder.0', 'encoder.2', 'head.fc1', 'head.fc2'], (
        f'unfreezing should add 2 more watched modules, got {watched2}'
    )
    assert wandb.watch.call_count == 4
    # log_freq propagates the new value.
    for call in wandb.watch.call_args_list:
        assert call.kwargs.get('log_freq') == 100

    # === Fully-frozen model → empty watched list, zero wandb.watch calls ===
    wandb.watch.reset_mock()
    for p in model.parameters():
        p.requires_grad_(False)
    watched3 = ex2_watch_trainable_modules(model, log_freq=50)
    assert watched3 == [], f'all-frozen model should yield empty list, got {watched3}'
    assert wandb.watch.call_count == 0

    # === Root module ('') excluded even if it has own params ===
    # Build a model where the root itself has a Parameter directly attached.
    wandb.watch.reset_mock()
    class WithRootParam(nn.Module):
        def __init__(self):
            super().__init__()
            self.bias = nn.Parameter(t.zeros(4))
            self.fc = nn.Linear(4, 4)
    m2 = WithRootParam()
    watched4 = ex2_watch_trainable_modules(m2, log_freq=10)
    # Even though root has a trainable own-param, qname=='' is filtered.
    assert '' not in watched4, 'root qname must be filtered'
    assert watched4 == ['fc'], f'expected ["fc"], got {watched4}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb
import torch.nn as nn

def ex2_watch_trainable_modules(model, log_freq):
    watched = []
    for qname, m in model.named_modules():
        if qname == '':
            continue
        own = list(m.parameters(recurse=False))
        if not own:
            continue
        if any(p.requires_grad for p in own):
            wandb.watch(m, log='all', log_freq=log_freq)
            watched.append(qname)
    return sorted(watched)
```

**`recurse=False` is the structural choice.** Without it, every ancestor of a trainable leaf reports `requires_grad=True` and gets a hook — you'd double- or triple-hook the same tensor histograms. `recurse=False` walks down to leaves and the wrapper modules drop out naturally.

**Why `any` not `all`.** A LayerNorm with frozen weight but trainable bias should still be watched — `any` captures the 'at least one trainable scalar' semantic. `all` would miss partially-frozen modules, which is the realistic case in fine-tuning.

**Sorted output for stable tests.** `named_modules` walks in insertion order, which is reproducible but order-dependent on model construction. Sorting the output makes the watched-list diffable across model edits.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()